Notebbok para el entramiento del modelo.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import sqlite3

ImportError: cannot import name 'Scaler' from 'sklearn.pipeline' (/home/jdd/workshop3-happiness-streaming/venv/lib/python3.12/site-packages/sklearn/pipeline.py)

In [3]:
df = pd.read_csv('../data/processed/df_clean.csv')

In [4]:
categorical_cols = ['Country']  # Única columna categórica
numerical_cols = [
    'Health (Life Expectancy)', 
    'Freedom', 
    'Generosity', 
    'Trust (Government Corruption)', 
    'Family', 
    'Economy (GDP per Capita)'
]  # Columnas numéricas relevantes
target_col = 'Happiness Score'  # Variable objetivo

# Nota: Excluimos 'Happiness Rank' porque probablemente está derivado de 'Happiness Score'
# Excluimos 'Year' porque no parece relevante para la predicción (puedes incluirlo si el EDA lo justifica)

# 4. Separar características (X) y variable objetivo (y)
X = df[categorical_cols + numerical_cols]
y = df[target_col]

# 5. Dividir en entrenamiento y prueba (70-30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 6. Preprocesamiento
# 6.1. Escalar columnas numéricas
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_test_num = scaler.transform(X_test[numerical_cols])

# Convertir a DataFrame para mantener los nombres de las columnas
X_train_num_df = pd.DataFrame(X_train_num, columns=numerical_cols, index=X_train.index)
X_test_num_df = pd.DataFrame(X_test_num, columns=numerical_cols, index=X_test.index)

# 6.2. Aplicar OneHotEncoder a la columna 'Country'
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

# Obtener nombres de las columnas codificadas
encoded_cols = encoder.get_feature_names_out(categorical_cols)
X_train_cat_df = pd.DataFrame(X_train_cat, columns=encoded_cols, index=X_train.index)
X_test_cat_df = pd.DataFrame(X_test_cat, columns=encoded_cols, index=X_test.index)

# 6.3. Combinar columnas numéricas escaladas y categóricas codificadas
X_train_encoded = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_encoded = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

# Verificar el dataset transformado
print("\nDataset de entrenamiento transformado:")
print(X_train_encoded.head())

# 7. Entrenar el modelo (Random Forest como ejemplo)
model = RandomForestRegressor(random_state=42)
model.fit(X_train_encoded, y_train)


y_pred = model.predict(X_test_encoded)

# 10. Evaluar el modelo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nMean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")



NameError: name 'StandardScaler' is not defined